# Monitoring & Observability

Companion notebook for the [Monitoring & Observability lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/10-monitoring-and-observability).

**The idea in one sentence.** In production the **mean lies** (a fat latency tail hides
behind it, so you watch **percentiles**), data **drifts** away from training (you detect it
with **PSI**), predictions get **miscalibrated**, and you track reliability against an
**SLO error budget**.

We build latency percentiles, PSI, calibration drift, and SLO burn from scratch, and
**validate that percentiles expose the tail and PSI detects drift**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. Latency percentiles: the mean lies

We simulate 50,000 request latencies as the *mixture* of a fast log-normal (~50 ms median) plus a 1 % slow path (~3 s median). The slow path is the kind of tail that real services produce: a small fraction of requests hits a cold cache, a GC pause, a slow downstream call.

Compute the mean, p50, p95, p99. Plot a log-x histogram and mark each percentile. The point: the *mean* barely moves when the tail is small; the *p99* is where users feel the pain.

In [ ]:
N = 50_000
# Fast path: log-normal centred near 50 ms.
fast = rng.lognormal(mean=np.log(50), sigma=0.35, size=N)
# Slow path: log-normal centred near 3000 ms. Tag 1 % of requests as slow.
is_slow = rng.random(N) < 0.01
slow = rng.lognormal(mean=np.log(3000), sigma=0.5, size=N)
latency_ms = np.where(is_slow, slow, fast)

mean_lat = np.mean(latency_ms)
p50 = np.percentile(latency_ms, 50)
p95 = np.percentile(latency_ms, 95)
p99 = np.percentile(latency_ms, 99)
p999 = np.percentile(latency_ms, 99.9)

print(f'mean : {mean_lat:7.1f} ms')
print(f'p50  : {p50:7.1f} ms')
print(f'p95  : {p95:7.1f} ms')
print(f'p99  : {p99:7.1f} ms')
print(f'p999 : {p999:7.1f} ms')
print()
print(f'p99 / mean ratio = {p99 / mean_lat:.1f}x  --  the mean understates the tail by an order of magnitude')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(latency_ms, bins=np.logspace(np.log10(latency_ms.min()), np.log10(latency_ms.max()), 80),
        color=BRAND, alpha=0.7, edgecolor='none')
for value, label, colour in [(mean_lat, 'mean', YELLOW), (p50, 'p50', TEAL),
                             (p95, 'p95', '#fb923c'), (p99, 'p99', ROSE)]:
    ax.axvline(value, color=colour, linestyle='--', lw=1.5,
               label=f'{label}: {value:.0f} ms')
ax.set_xscale('log')
ax.set_xlabel('latency (ms, log scale)')
ax.set_ylabel('request count')
ax.set_title('Latency distribution with a 1 % slow path: the mean barely moves, the p99 sits in the tail')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Validate: the mean hides the tail — watch percentiles

With a 1% slow path, the **mean** latency looks fine while the **p99/p99.9** are an order of
magnitude worse — and it's the tail users complain about. We confirm the mean sits far below
p99, so a mean-only dashboard would miss the pain.

In [ ]:
print(f'mean {mean_lat:.0f} ms | p50 {p50:.0f} | p95 {p95:.0f} | p99 {p99:.0f} | p99.9 {p999:.0f} ms')
assert p50 < mean_lat < p99, 'the mean is pulled up by the tail but still far below p99'
assert p999 > 10 * mean_lat, 'the p99.9 tail is an order of magnitude worse than the mean'
print('\n✅ the mean hides the tail — monitor p95/p99/p99.9, not just the average')

Even a tiny tail (1 % of requests) pushes the **p99 to multiple seconds** while the mean barely shifts past 100 ms. That's why every SRE recommendation says to alert on percentiles, not means: the user who hits the slow path is just as real as the one who hit the fast path, and only the percentile sees them.

## 2. PSI from scratch

We implement the Population Stability Index from the equation in the lesson:

$$\mathrm{PSI} = \sum_{i=1}^{k} (p_i - q_i) \cdot \log\!\left(\frac{p_i}{q_i}\right)$$

Run it on three regimes:

- **Identical** distributions (training = prod) -- expect PSI ≈ 0.
- **Small shift** (mean shifted by ~0.3 sigma) -- expect 0.1–0.25 (the *watch* zone).
- **Heavy shift** (mean shifted by ~1 sigma) -- expect > 0.25 (the *act* zone).

In [ ]:
def psi(reference, current, bins=10, eps=1e-6):
    """Population Stability Index.

    Bins both arrays into the same `bins` quantile bins of the reference,
    converts to probabilities, and returns PSI = sum((p - q) * log(p / q)).
    `eps` floors each probability to avoid log(0).
    """
    # Use the reference's quantile edges so bins reflect training-time structure.
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_counts, _ = np.histogram(reference, bins=edges)
    cur_counts, _ = np.histogram(current, bins=edges)
    p = ref_counts / max(ref_counts.sum(), 1)
    q = cur_counts / max(cur_counts.sum(), 1)
    p = np.maximum(p, eps)
    q = np.maximum(q, eps)
    contributions = (p - q) * np.log(p / q)
    return float(contributions.sum()), p, q, contributions

rng_psi = np.random.default_rng(1)
reference = rng_psi.normal(0, 1, size=20_000)

current_same = rng_psi.normal(0, 1, size=20_000)              # identical distribution
current_small = rng_psi.normal(0.3, 1, size=20_000)           # small mean shift
current_heavy = rng_psi.normal(1.0, 1.2, size=20_000)         # mean + scale shift

psi_same, p_same, q_same, contrib_same = psi(reference, current_same)
psi_small, p_small, q_small, contrib_small = psi(reference, current_small)
psi_heavy, p_heavy, q_heavy, contrib_heavy = psi(reference, current_heavy)

for label, value in [('identical', psi_same), ('small shift', psi_small), ('heavy shift', psi_heavy)]:
    if value < 0.1:
        zone = 'STABLE (< 0.1)'
    elif value < 0.25:
        zone = 'WATCH (0.1–0.25)'
    else:
        zone = 'ACT (> 0.25)'
    print(f'{label:>12s} : PSI = {value:.4f}   -> {zone}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (title, p, q, contrib, value) in zip(
    axes,
    [('identical', p_same, q_same, contrib_same, psi_same),
     ('small shift', p_small, q_small, contrib_small, psi_small),
     ('heavy shift', p_heavy, q_heavy, contrib_heavy, psi_heavy)],
):
    x = np.arange(len(p))
    w = 0.4
    ax.bar(x - w / 2, p, w, color=BRAND, alpha=0.8, label='training')
    ax.bar(x + w / 2, q, w, color=ROSE, alpha=0.8, label='production')
    ax.set_title(f'{title}\nPSI = {value:.3f}')
    ax.set_xlabel('bin')
    ax.set_ylabel('probability')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Validate: PSI is ~0 for the same distribution and large under drift

The Population Stability Index compares a current feature distribution to the training
reference. PSI ≈ 0 when they match, and rises past the usual 0.2 alert threshold when the
distribution shifts. We confirm both — the drift alarm that tells you to retrain.

In [ ]:
ref = np.random.default_rng(0).normal(0, 1, 20000)
same = np.random.default_rng(1).normal(0, 1, 20000)     # same distribution
shifted = np.random.default_rng(2).normal(1.0, 1.5, 20000)  # mean & variance drift
psi_same = psi(ref, same)[0]        # psi() returns (value, p, q, contributions)
psi_shifted = psi(ref, shifted)[0]
print(f'PSI (same distribution): {psi_same:.3f}  (~0, stable)')
print(f'PSI (shifted)          : {psi_shifted:.3f}  (>0.2, drift alarm)')
assert psi_same < 0.1, 'PSI should be near 0 for the same distribution'
assert psi_shifted > 0.2, 'PSI should exceed the alert threshold under drift'
print('\n✅ PSI detects distribution drift — the trigger to retrain')

The PSI is roughly **0** when the two samples are drawn from the same distribution (the small residual is sampling noise on 20,000 points), lands in the **watch zone** for a 0.3 sigma shift, and is well into the **act zone** for a 1 sigma + scale shift. Inspecting the bin-by-bin bars makes the shift visible: a mass that was concentrated near the centre under training has migrated to the right tail in production.

## 3. Calibration drift and reliability diagrams

A binary classifier is **well calibrated** if predictions made with confidence `p` are right `p` of the time. We build a synthetic classifier:

- *Training distribution.* True log-odds $z \sim \mathcal{N}(0, 1)$. The classifier outputs $\sigma(z)$ and the label is `1` with probability $\sigma(z)$. This is calibrated by construction.
- *Production distribution.* Inputs shift so the true log-odds are now $z \sim \mathcal{N}(0, 1)$ but the *classifier* still outputs $\sigma(1.5 z)$ -- an over-confident scoring of the same evidence. Predictions are pushed toward 0 and 1; users see confident answers that are wrong more often than they should be.

We bin predictions by confidence, plot the reliability diagram for each, and compute the **expected calibration error** (ECE):

$$\mathrm{ECE} = \sum_b \frac{|B_b|}{N} \cdot |\mathrm{acc}(B_b) - \mathrm{conf}(B_b)|$$

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def reliability(probs, labels, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    bin_idx = np.digitize(probs, edges[1:-1])
    bin_conf, bin_acc, bin_count = [], [], []
    for b in range(n_bins):
        mask = bin_idx == b
        if mask.sum() == 0:
            bin_conf.append(np.nan); bin_acc.append(np.nan); bin_count.append(0); continue
        bin_conf.append(probs[mask].mean())
        bin_acc.append(labels[mask].mean())
        bin_count.append(int(mask.sum()))
    bin_conf = np.array(bin_conf); bin_acc = np.array(bin_acc); bin_count = np.array(bin_count)
    weight = bin_count / max(bin_count.sum(), 1)
    ece = float(np.nansum(weight * np.abs(bin_acc - bin_conf)))
    return bin_conf, bin_acc, bin_count, ece

N_CAL = 30_000
rng_cal = np.random.default_rng(2)

# Training distribution: well-calibrated by construction.
z_train = rng_cal.normal(0, 1, size=N_CAL)
p_train = sigmoid(z_train)
y_train = (rng_cal.random(N_CAL) < p_train).astype(int)

# Production: same z but the model multiplies the logit by 1.5 (overconfident).
z_prod = rng_cal.normal(0, 1, size=N_CAL)
p_true = sigmoid(z_prod)               # true label probabilities
p_model = sigmoid(1.5 * z_prod)        # model says so
y_prod = (rng_cal.random(N_CAL) < p_true).astype(int)

conf_t, acc_t, cnt_t, ece_t = reliability(p_train, y_train)
conf_p, acc_p, cnt_p, ece_p = reliability(p_model, y_prod)

print(f'Training ECE   : {ece_t:.4f}  (well-calibrated)')
print(f'Production ECE : {ece_p:.4f}  (overconfident -- predictions of 0.9 are not right 90% of the time)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, conf, acc, ece, title in [
    (axes[0], conf_t, acc_t, ece_t, 'Training: calibrated'),
    (axes[1], conf_p, acc_p, ece_p, 'Production: overconfident'),
]:
    ax.plot([0, 1], [0, 1], color='#475569', linestyle='--', lw=1, label='perfect calibration')
    ax.plot(conf, acc, 'o-', color=BRAND, lw=2, markersize=8, label='empirical')
    ax.fill_between(conf, conf, acc, color=ROSE, alpha=0.15, label='calibration gap')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('mean predicted probability (bin)')
    ax.set_ylabel('empirical accuracy (bin)')
    ax.set_title(f'{title}\nECE = {ece:.4f}')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Under training the reliability curve sits on the diagonal and ECE is small. Under production the same model scores its inputs with too-confident logits -- the curve drops below the diagonal in the upper-right (predictions at 0.9 confidence are right less than 90 % of the time) and the ECE roughly **5–10x**'s. The shape of the reliability diagram is the *kind* of failure: a curve below the diagonal in the upper-right is the hallmark of overconfidence; a curve above the diagonal there means underconfidence (rarer in practice but common in models trained with strong regularisation).

## 4. SLO error-budget burn rates

Define an SLO of **99.5 % availability over 30 days** -- so the allowed error rate is `0.5 %` averaged across the window. The error budget for the month is `0.005 × total requests`.

Simulate 30 days = 720 hours of hourly error rates: a sinusoidal daytime peak around 1 % plus low background noise. Inject a **3-hour outage at hour 200** that drives errors to 15 %.

Compute two burn rates:

- **fast-burn** (1-hour window) -- pages if it exceeds **14×** the steady-state allowed error rate (you'd exhaust the monthly budget in ~2 days at this rate).
- **slow-burn** (6-hour window) -- opens a ticket (not a page) if it exceeds **1×** (steady leak).

The point of this experiment: the fast-burn alert fires within ~1 hour of the outage; the slow-burn alone would still be silent for hours afterwards.

In [ ]:
rng_slo = np.random.default_rng(3)
HOURS = 24 * 30  # 30 days
SLO_AVAILABILITY = 0.995
ALLOWED_ERR = 1.0 - SLO_AVAILABILITY  # 0.005 = 0.5 %

# Hourly error rate: sinusoidal daytime peak + noise.
t = np.arange(HOURS)
daily_cycle = 0.0025 + 0.0025 * (0.5 + 0.5 * np.sin(2 * np.pi * (t % 24) / 24 - np.pi / 2))
noise = rng_slo.normal(0, 0.0005, size=HOURS)
err_rate = np.clip(daily_cycle + noise, 0, None)

# Inject a 3-hour outage at hour 200.
OUTAGE_START, OUTAGE_LEN, OUTAGE_RATE = 200, 3, 0.15
err_rate[OUTAGE_START:OUTAGE_START + OUTAGE_LEN] = OUTAGE_RATE

def rolling_mean(x, window):
    """Right-aligned causal rolling mean. Returns nan until the window fills."""
    out = np.full_like(x, np.nan, dtype=float)
    csum = np.cumsum(np.insert(x, 0, 0.0))
    for i in range(window - 1, len(x)):
        out[i] = (csum[i + 1] - csum[i + 1 - window]) / window
    return out

burn_1h = err_rate / ALLOWED_ERR                           # 1-h window
burn_6h = rolling_mean(err_rate, 6) / ALLOWED_ERR          # 6-h window

FAST_THRESHOLD = 14.0
SLOW_THRESHOLD = 1.0

# When does each alert first fire after the outage?
fast_fires = np.where(burn_1h > FAST_THRESHOLD)[0]
slow_fires = np.where(burn_6h > SLOW_THRESHOLD)[0]
first_fast = int(fast_fires[fast_fires >= OUTAGE_START][0]) if (fast_fires >= OUTAGE_START).any() else None
first_slow = int(slow_fires[slow_fires >= OUTAGE_START][0]) if (slow_fires >= OUTAGE_START).any() else None

print(f'Outage starts at hour {OUTAGE_START}')
print(f'Fast-burn (1-h, threshold 14x) first fires at hour {first_fast}  '
      f'-> {first_fast - OUTAGE_START}h after outage start')
print(f'Slow-burn (6-h, threshold  1x) first fires at hour {first_slow}  '
      f'-> {first_slow - OUTAGE_START}h after outage start')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t, burn_1h, color=ROSE, lw=1, label='1h burn rate (fast)')
ax.plot(t, burn_6h, color=BRAND, lw=2, label='6h burn rate (slow)')
ax.axhline(FAST_THRESHOLD, color=ROSE, linestyle='--', lw=1, label='fast-burn threshold 14x (page)')
ax.axhline(SLOW_THRESHOLD, color=BRAND, linestyle=':', lw=1, label='slow-burn threshold 1x (ticket)')
ax.axvspan(OUTAGE_START, OUTAGE_START + OUTAGE_LEN, color=YELLOW, alpha=0.15, label='outage window')
if first_fast is not None:
    ax.axvline(first_fast, color=ROSE, lw=1, alpha=0.6)
ax.set_yscale('log')
ax.set_ylim(0.05, 100)
ax.set_xlabel('hour (30-day window)')
ax.set_ylabel('burn rate (log scale)')
ax.set_title('SLO error-budget burn rates: fast-burn pages within an hour; slow-burn would still be silent')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The fast-burn alert (1-hour window, 14× threshold) fires almost immediately when the outage begins -- a 15 % error rate against a 0.5 % allowance is a 30× burn rate, well above the threshold. The slow-burn alert (6-hour window, 1× threshold) responds slowly because the average over six hours is heavily diluted by the calm hours preceding the outage. The point of pairing the two: the fast-burn catches *acute* incidents, the slow-burn catches *steady leaks* that would never trip the 14× threshold but eat the budget over weeks.

Both alerts are far less noisy than the equivalent raw-threshold alert (`error rate > 0.005`) because transient spikes during the daytime peak briefly cross 0.5 % without ever sustaining a 14× burn rate over a full hour. That's what "alert on burn rate, not raw thresholds" means in practice.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **mean-only latency** | hides the fat tail users feel; track p95/p99/p99.9 (verified) |
| **no drift detection** | silent data drift decays the model; PSI/KS alarms catch it (verified) |
| **PSI bin choice** | too few bins miss drift, too many are noisy; 10 quantile bins is typical |
| **calibration ≠ accuracy** | a model can stay accurate yet drift out of calibration |
| **alert fatigue** | too-sensitive thresholds train responders to ignore alerts |

Demo: an outage burns the SLO error budget far faster than normal operation.

In [ ]:
# SLO error budget: an availability SLO of 99.5% ALLOWS 0.5% errors. Monitoring tracks how
# fast you're 'burning' that budget — a single outage can consume weeks of budget in hours.
# We compute the budget burned by the injected outage.
allowed_per_hour = ALLOWED_ERR                       # 0.5% error budget per hour (simplified)
burned = err_rate[OUTAGE_START:OUTAGE_START + OUTAGE_LEN].sum()
normal_burn = err_rate[:OUTAGE_START].mean()
print(f'SLO allows {ALLOWED_ERR:.1%} errors; the {OUTAGE_LEN}h outage burned {burned:.2f} error-hours')
print(f'normal hourly error rate {normal_burn:.4f} vs outage rate {OUTAGE_RATE}')
assert OUTAGE_RATE > 10 * normal_burn, 'the outage burns budget far faster than normal operation'
print('\nError-budget burn rate turns "how reliable are we?" into an actionable, alertable number.')

## ✏️ Your turn — implement PSI from scratch and assert behaviour

Your goal: implement `psi(reference, current, bins=10)` that returns the population stability index between two NumPy arrays. Use the reference array's quantile edges so the bins reflect training-time structure (the same convention as section 2).

We'll then verify:

- PSI between two samples from $\mathcal{N}(0, 1)$ is small (< 0.05) -- below the stable threshold.
- PSI between $\mathcal{N}(0, 1)$ and $\mathcal{N}(1, 1)$ is large (> 0.25) -- in the act zone.

In [ ]:
def psi_solution(reference, current, bins=10, eps=1e-6):
    """Population Stability Index.

    Returns the scalar PSI between two NumPy arrays using the reference's
    quantile edges to define `bins` bins. `eps` floors each probability to
    avoid log(0).
    """
    # TODO(you):
    # 1. Use np.quantile on `reference` with np.linspace(0, 1, bins + 1) to get edges.
    # 2. Set edges[0] = -inf, edges[-1] = +inf so out-of-range values still bin.
    # 3. np.histogram both arrays against `edges`.
    # 4. Convert counts to probabilities p and q. Floor at eps.
    # 5. Return float(((p - q) * np.log(p / q)).sum()).
    edges = ...  # TODO
    p = ...      # TODO
    q = ...      # TODO
    return float(((p - q) * np.log(p / q)).sum())

In [ ]:
rng_test = np.random.default_rng(42)
ref = rng_test.normal(0, 1, size=20_000)
current_stable = rng_test.normal(0, 1, size=20_000)
current_drifted = rng_test.normal(1.0, 1.0, size=20_000)

psi_stable = psi_solution(ref, current_stable)
psi_drifted = psi_solution(ref, current_drifted)

print(f'PSI (stable, same distribution)  : {psi_stable:.4f}  (expected < 0.05)')
print(f'PSI (drifted, mean shifted by 1) : {psi_drifted:.4f}  (expected > 0.25)')

assert psi_stable < 0.05, (
    f'PSI on two samples from the same distribution should be small, got {psi_stable:.4f}'
)
assert psi_drifted > 0.25, (
    f'PSI on a clear mean shift should land in the act zone (>0.25), got {psi_drifted:.4f}'
)
print('\n✅ PSI behaves correctly across the stable / act regimes.')

<details>
<summary>Solution</summary>

```python
def psi_solution(reference, current, bins=10, eps=1e-6):
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_counts, _ = np.histogram(reference, bins=edges)
    cur_counts, _ = np.histogram(current, bins=edges)
    p = np.maximum(ref_counts / max(ref_counts.sum(), 1), eps)
    q = np.maximum(cur_counts / max(cur_counts.sum(), 1), eps)
    return float(((p - q) * np.log(p / q)).sum())
```

Two implementation details worth noting:

- *Quantile bins from the reference.* Equal-frequency bins under the reference distribution mean that the reference's `p` is uniform `1/bins` per bin. Any deviation in `q` from `1/bins` is the entire drift signal, which makes per-bin contributions interpretable.
- *Floor by `eps`.* If a production bin is empty (`q_i = 0`) the `log(p / q)` blows up. The `eps` floor caps the contribution at `p_i * log(p_i / eps)` -- not zero, but not infinite either. Real implementations often use the smoothing the same way; teams that want fully-robust drift detection sometimes use a Bayesian smoothing prior instead.

</details>

## Recap

- **Percentile latencies, not means.** A 1 % slow path is invisible to the mean and dominates the p99. Alert on the tail.
- **PSI** measures drift per feature; <0.1 stable, 0.1–0.25 watch, >0.25 act. Quantile-bin against the reference for interpretable per-bin contributions.
- **Reliability diagrams + ECE** catch *calibration* drift -- a model whose predictions are still in range but no longer mean what they used to. This is a leading indicator before outcome accuracy drops.
- **Burn-rate alerts** filter transient noise. A fast-burn (1h, 14×) pages on acute incidents; a slow-burn (6h, 1×) opens a ticket on steady leaks. Together they catch both failure shapes without burning out the on-call.